Imports

In [1]:
import os
import pandas as pd
import joblib

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

Define paths & load split datasets

In [2]:
# Project root (adjust if needed)
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
data_path = os.path.join(project_root, "data", "processed", "split")

# Training & Twitter test split
X_train = pd.read_csv(os.path.join(data_path, "X_train.csv"))["clean_text"]
y_train = pd.read_csv(os.path.join(data_path, "y_train.csv"))["label_encoded"]

X_test = pd.read_csv(os.path.join(data_path, "X_test.csv"))["clean_text"]
y_test = pd.read_csv(os.path.join(data_path, "y_test.csv"))["label_encoded"]

# Cross-platform test split (clean_text only)
X_test_cross = pd.read_csv(os.path.join(data_path, "X_test_cross.csv"))["clean_text"]
y_test_cross = pd.read_csv(os.path.join(data_path, "y_test_cross.csv"))["label_encoded"]

print("✅ Data loaded successfully")

✅ Data loaded successfully


TF-IDF Vectorization

In [3]:
# Handle missing / empty text 
# TfidfVectorizer cannot process NaN, it only accepts strings

# Fix NaN values (very important)
X_train = X_train.fillna("")
X_test = X_test.fillna("")
X_test_cross = X_test_cross.fillna("")

print("✅ NaN values handled")

✅ NaN values handled


In [4]:
# Feature extraction step for Logistic Regression
tfidf = TfidfVectorizer(
    lowercase=False,
    strip_accents="unicode",
    max_features=25000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
X_test_cross_tfidf = tfidf.transform(X_test_cross)

print("✅ TF-IDF vectorization completed")

✅ TF-IDF vectorization completed


Train Logistic Regression model

In [5]:
lr_model = LogisticRegression(
    C=2.0,
    penalty="l2",
    solver="liblinear",
    class_weight="balanced",
    max_iter=3000,
    random_state=42
)
lr_model.fit(X_train_tfidf, y_train)

print("✅ Logistic Regression model trained")

✅ Logistic Regression model trained


In [6]:
print("Training Accuracy:",
      lr_model.score(X_train_tfidf, y_train))

print("Testing Accuracy:",
      lr_model.score(X_test_tfidf, y_test))

Training Accuracy: 0.9153413431488298
Testing Accuracy: 0.8660833637136954


Evaluation on Kaggle test split 

In [7]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("📊 Twitter Test Split Results")

# Predicted class labels
y_pred = lr_model.predict(X_test_tfidf)

# Predicted probabilities for the positive class
y_prob = lr_model.predict_proba(X_test_tfidf)[:, 1]

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Classification Report
print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Cyberbullying", "Cyberbullying"]
))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix")
print(cm)

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_prob)
print(f"\nROC-AUC Score: {roc_auc:.4f}")

📊 Twitter Test Split Results
Accuracy: 0.8661

Classification Report
                   precision    recall  f1-score   support

Not Cyberbullying       0.48      0.78      0.60      1039
    Cyberbullying       0.97      0.88      0.92      7190

         accuracy                           0.87      8229
        macro avg       0.72      0.83      0.76      8229
     weighted avg       0.90      0.87      0.88      8229


Confusion Matrix
[[ 812  227]
 [ 875 6315]]

ROC-AUC Score: 0.9216


Evaluation on Cross-Platform dataset

In [ ]:
print("🌍 Cross-Platform Test Results")

# Predicted probabilities
y_prob_cross = lr_model.predict_proba(X_test_cross_tfidf)[:, 1]

# Decision threshold
threshold = 0.42        #the results dont change from 40-42

# Convert probabilities to class labels
y_pred_cross = (y_prob_cross >= threshold).astype(int)

# Accuracy
accuracy_cross = accuracy_score(y_test_cross, y_pred_cross)
print(f"Accuracy: {accuracy_cross:.4f}")

# Classification Report
print("\nClassification Report")
print(classification_report(
    y_test_cross,
    y_pred_cross,
    target_names=["Not Cyberbullying", "Cyberbullying"]
))

# Confusion Matrix
cm_cross = confusion_matrix(y_test_cross, y_pred_cross)
print("\nConfusion Matrix")
print(cm_cross)

# ROC-AUC Score
roc_auc_cross = roc_auc_score(y_test_cross, y_prob_cross)
print(f"\nROC-AUC Score: {roc_auc_cross:.4f}")

🌍 Cross-Platform Test Results
Accuracy: 0.8105

Classification Report
                   precision    recall  f1-score   support

Not Cyberbullying       0.77      0.91      0.83      1173
    Cyberbullying       0.88      0.70      0.78      1085

         accuracy                           0.81      2258
        macro avg       0.82      0.81      0.81      2258
     weighted avg       0.82      0.81      0.81      2258


Confusion Matrix
[[1070  103]
 [ 325  760]]

ROC-AUC Score: 0.8908


platform-wise evaluation

In [9]:
# Load new platform-aware test file
df_cross_test = pd.read_csv(os.path.join(data_path, "X_test_cross_full.csv"))

print("\n📊 Platform-wise Cross-Platform Test Results")

for platform in df_cross_test['Platform'].unique():
    print(f"\n🔹 Platform: {platform}")

    # Filter platform-specific data
    df_platform = df_cross_test[df_cross_test['Platform'] == platform]

    # Transform text using TF-IDF
    X_platform_tfidf = tfidf.transform(df_platform['clean_text'].fillna(""))
    y_platform = df_platform['label_encoded']

    # Predictions
    y_pred_platform = lr_model.predict(X_platform_tfidf)

    # Metrics
    print("Accuracy:", accuracy_score(y_platform, y_pred_platform))
    print(classification_report(
        y_platform,
        y_pred_platform,
        target_names=["Not Cyberbullying", "Cyberbullying"]
    ))


📊 Platform-wise Cross-Platform Test Results

🔹 Platform: Reddit
Accuracy: 0.7100671140939597
                   precision    recall  f1-score   support

Not Cyberbullying       0.63      1.00      0.78       374
    Cyberbullying       1.00      0.42      0.59       371

         accuracy                           0.71       745
        macro avg       0.82      0.71      0.68       745
     weighted avg       0.82      0.71      0.68       745


🔹 Platform: Facebook
Accuracy: 0.7077326343381389
                   precision    recall  f1-score   support

Not Cyberbullying       0.64      1.00      0.78       392
    Cyberbullying       1.00      0.40      0.57       371

         accuracy                           0.71       763
        macro avg       0.82      0.70      0.67       763
     weighted avg       0.81      0.71      0.68       763


🔹 Platform: Instagram
Accuracy: 0.736
                   precision    recall  f1-score   support

Not Cyberbullying       0.67      1.00    

Save model & vectorizer

In [10]:
model_path = os.path.join(project_root, "models", "baseline")
os.makedirs(model_path, exist_ok=True)

joblib.dump(lr_model, os.path.join(model_path, "logistic_regression_model.pkl"))
joblib.dump(tfidf, os.path.join(model_path, "tfidf_vectorizer.pkl"))

print("✅ Model and TF-IDF vectorizer saved")

✅ Model and TF-IDF vectorizer saved
